> **The image problem:** UnifiedAI's property-assessment platform processes 40,000 paper forms per year — each form has a handwritten 4-digit property code in the top-right corner, written by field agents under time pressure. The company has a one-time labelling budget: **2,000 scanned digit images**, no more.
>
> The dense MLP from P-2 hits 97% on perfectly centred MNIST digits — but real field agents don't centre their digits. On actual scanned forms, shifted or slightly rotated handwriting drops the MLP to **71% accuracy**: one digit wrong in every three property codes, cascading into incorrect tax assessments.
>
> **Root cause:** the MLP treats pixel (3, 5) and pixel (3, 8) as completely independent features. If a "3" shifts 3 pixels right, the model has never seen that exact pixel pattern and fails. A **convolution** shares the same detector weights at every position — if you can detect a horizontal arc at position (3, 5), you automatically detect it at (3, 8) without extra training.
>
> **This chapter's task:** build a CNN from scratch, verify every design decision with a measurement (not an assertion), and end with a transfer-learning solution that reaches >95% on the 2,000-example budget.


# P-3 · Convolutional Neural Networks

**Track:** genai-prerequisites — zero-to-LLM-ready  
**Position:** follows P-2 (neural networks + backprop on XOR) · precedes P-4 (RNN sequence modeling)  
**Running example:** MNIST digit "3" — a handwritten digit from the same distribution as UnifiedAI's scanned property-code forms.

---

A dense MLP can reach ~98% on MNIST — so why bother with CNNs? Because the MLP only works when:

- Every pixel is always at the same location (no shift, scale, or rotation)
- You have enough labelled data to learn 784 independent weights per class
- You never need to generalize to natural images, video frames, or higher-resolution scans

CNNs break all three constraints by exploiting **translation equivariance** (the same edge looks like an edge wherever it appears) and **spatial locality** (nearby pixels are more related than distant ones). These two inductive biases slash the parameter count from millions to thousands and generalise far better from small datasets.

---

## Roadmap

| Part  | Concept                         | Key idea                                                        |
| ----- | ------------------------------- | --------------------------------------------------------------- |
| **1** | Convolution as a learned filter | A filter slides over the image detecting one pattern everywhere |
| **2** | Stride & pooling                | Spatial compression without losing what matters                 |
| **3** | Receptive field                 | Deeper layers see larger patches — complexity grows with depth  |
| **4** | ResNet skip connections         | Adding `+x` bypasses prevent vanishing gradients in deep nets   |
| **5** | Transfer learning               | Freeze ImageNet features; fine-tune only the head               |
| **6** | Toy → real bridge               | Our 30k-param CNN vs ResNet-18 vs ResNet-50 vs ViT-B/16         |


In [ ]:
#  Setup: install missing packages, import, seed, load MNIST 
import subprocess, sys

for pkg in ["torch", "torchvision", "numpy", "matplotlib"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt

# Deterministic seeds — every quoted number in this notebook reproduces exactly
torch.manual_seed(42)
np.random.seed(42)

# MNIST normalisation constants (mean=0.1307, std=0.3081 over all 60k training pixels)
transform = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])

train_ds = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.MNIST(
    "./data", train=False, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=64, shuffle=True)

print(f"MNIST: {len(train_ds):,} train, {len(test_ds):,} test | image shape: 1×28×28")

# Grab the first digit '3' from the training set — our running example throughout
for imgs, labels in train_loader:
    idx = (labels == 3).nonzero(as_tuple=True)[0]
    if len(idx):
        example = imgs[idx[0]]
        break

print(f"Running example: digit '3', tensor shape {tuple(example.shape)}")
print(
    f"Pixel value range after normalisation: [{example.min():.3f}, {example.max():.3f}]"
)

In [ ]:
#  Display running example + MLP vs CNN framing 
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(example.squeeze(), cmap="gray")
ax.set_title("Running example: MNIST '3'", fontsize=13, pad=10)
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Pixel range after normalisation: [{example.min():.2f}, {example.max():.2f}]")
print()
print("MLP approach: flatten 1×28×28 → vector of 784 numbers")
print("  → pixel (3,5) is treated as INDEPENDENT of pixel (3,6)")
print("  → 784 × 128 = 100,352 weights just for the first layer")
print()
print("CNN approach: slide a 3×3 filter over every local patch")
print("  → same 9 weights detect the same pattern anywhere in the image")
print("  → 3×3 × 1 channel × 32 filters = 288 weights for the first layer")
print("  → 350× fewer parameters, translation-equivariant by construction")

---

## Part 1 — Convolution as a Learned Filter

A convolution is just a **sliding dot product**: a small weight matrix (the *filter* or *kernel*) is placed over every position in the input and the dot product is computed. The result is a *feature map* — a new image showing where that filter's pattern was found.

The same weights are used at every position. That single decision gives CNNs **translation equivariance**: if the pattern moves 5 pixels to the right, the feature map shifts by the same amount — the detector doesn't need to be retrained.

Before CNNs, engineers hand-crafted filters (Sobel, Gabor, HOG). The CNN breakthrough: **learn the filters from data** via backprop. A trained CNN's first layer learns filters that look strikingly similar to Sobel/Gabor — but optimised for the specific task.

---

#### Predict first — before running the next cell:**

A Sobel horizontal-edge filter has weights:

```
[[-1, -2, -1],
 [ 0,  0,  0],
 [+1, +2, +1]]
```

It fires strongly where pixel intensity **increases sharply from top to bottom**.

When applied to the digit "3", what will be highlighted?

- **(a)** The horizontal curves at the top and bottom of the "3"
- **(b)** The vertical strokes
- **(c)** The whole digit uniformly

Pick your answer, then run the cell below to check.


![Convolution filter operation: a 3×3 filter slides over a 5×5 input patch, computing a dot product to produce one output feature map value](images/convolution-filter-operation.png)


In [ ]:
#  Single-pixel walkthrough: what one convolution output pixel means 
# Let's trace exactly one output pixel to demystify conv2d

# Take a 3×3 patch from the digit "3" at position (row=10, col=12)
patch = example.squeeze()[10:13, 12:15]  # 3×3 patch
filter_3x3 = torch.tensor(
    [[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]]
)  # Sobel horizontal

print("Patch of digit '3' at position (10-12, 12-14):")
print(patch.numpy().round(2))
print()
print("Sobel filter:")
print(filter_3x3.numpy())
print()

# Element-wise multiply then sum = one output pixel
product = patch * filter_3x3
output_pixel = product.sum().item()
print("Element-wise products:")
print(product.numpy().round(2))
print()
print(
    f"Sum of products = {output_pixel:.3f}  ← this is ONE output pixel in the feature map"
)
print("Positive = bright horizontal edge above, dark below. Negative = opposite.")

In [ ]:
#  Hand-coded Sobel filter applied to digit '3' 
# The Sobel horizontal filter detects horizontal edges (intensity change top→bottom)
sobel_h = torch.tensor([[-1.0, -2.0, -1.0], [0.0, 0.0, 0.0], [1.0, 2.0, 1.0]]).view(
    1, 1, 3, 3
)

img_b = example.unsqueeze(0)  # add batch dim: (1,1,28,28)

with torch.no_grad():
    edge = torch.nn.functional.conv2d(img_b, sobel_h, padding=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(example.squeeze(), cmap="gray")
axes[0].set_title("Original '3'", fontsize=12)
axes[0].axis("off")
axes[1].imshow(edge.squeeze().abs(), cmap="hot")
axes[1].set_title("Sobel filter output (horizontal edges)", fontsize=12)
axes[1].axis("off")
plt.suptitle(
    "One hand-designed filter — CNN learns hundreds of these automatically",
    fontsize=11,
    style="italic",
)
plt.tight_layout()
plt.show()

print(
    f"Input shape:  {tuple(img_b.shape)}   (batch=1, channels=1, height=28, width=28)"
)
print(
    f"Output shape: {tuple(edge.shape)}   (batch=1, 1 feature map, 28×28 — padding preserves size)"
)
print()
print("Prediction check: answer (a) ")
print("  → The TOP and BOTTOM arcs of the '3' are bright (high response)")
print(
    "  → Vertical strokes barely activate because there's no top-to-bottom intensity jump"
)
print()
print("Key insight: this filter was HAND-DESIGNED by an engineer.")
print("CNNs learn such filters automatically from labelled data via backprop.")

In [ ]:
#  Visualise 8 randomly-initialised learned filters 
# After training these look like Sobel / Gabor / corner detectors — specialised by data
torch.manual_seed(42)
conv = nn.Conv2d(1, 8, kernel_size=3, padding=1)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    w = conv.weight[i, 0].detach().numpy()
    ax.imshow(w, cmap="RdBu_r", vmin=-0.5, vmax=0.5)
    ax.set_title(f"Filter {i+1}", fontsize=10)
    ax.axis("off")
plt.suptitle("8 learned conv filters — random init (before training)", fontsize=12)
plt.tight_layout()
plt.show()

n_params = 8 * 1 * 3 * 3  # out_channels × in_channels × kernel_H × kernel_W
print(f"8 filters × 1 input channel × 3×3 = {n_params} trainable weights")
print(
    f"Compare: MLP first hidden layer of 128 neurons = 784 × 128 = {784*128:,} weights"
)
print(f"  → CNN first layer is {784*128 // n_params}× smaller — same spatial coverage")
print()
print("After training on MNIST, each filter specialises for one pattern:")
print("  Filter 1 → horizontal edges   Filter 2 → vertical edges")
print("  Filter 3 → diagonal /         Filter 4 → diagonal \\")
print("  Filter 5 → top-left corner    ...and so on")
print()
print("#### Your turn — change kernel_size=3 to kernel_size=5 and re-run.")
print("   # # CHANGE kernel_size above. Do filters look more or less structured?")

---

## Part 2 — Stride & Pooling

A conv layer with `padding=1` and `stride=1` produces a feature map the **same size** as the input. For a 28×28 image through 10 conv layers this would be expensive and redundant — nearby positions share almost identical context.

Two standard tools **reduce spatial resolution** while keeping the important information:

| Mechanism        | How it works                              | When to use                                            |
| ---------------- | ----------------------------------------- | ------------------------------------------------------ |
| **Stride > 1**   | Step the filter by S pixels instead of 1  | When you want the conv itself to downsample            |
| **MaxPool2d(k)** | Keep the maximum value in each k×k window | After a conv when you want to retain sharp activations |
| **AvgPool2d(k)** | Keep the average value in each k×k window | Global pooling before a classifier head                |

### Output size formula

For an input of width $W$, kernel size $K$, padding $P$, stride $S$:

$$W_{out} = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

Plain English: "how many times can the kernel slide across the (padded) input?" Add 1 for the starting position.

**Example:** 28×28 input, 3×3 kernel, padding=1, stride=1 → $\lfloor (28-3+2) / 1 \rfloor + 1 = 28$ (size preserved).  
**With MaxPool 2×2, stride=2:** $\lfloor (28-2+0) / 2 \rfloor + 1 = 14$ (halved).


![Feature maps by layer: raw MNIST digit → Conv1 edge detectors → Conv2 abstract patterns at 7×7](images/feature-maps-by-layer.png)


In [ ]:
#  Max-pooling visual diagram 
import matplotlib.patches as mpatches

# 4×4 input — windows chosen so maxima are 6, 4, 5, 3
grid = np.array([[1, 3, 2, 4], [5, 6, 1, 2], [3, 5, 0, 1], [1, 2, 3, 0]], dtype=float)

COLOURS = ["#4CAF50", "#2196F3", "#FF9800", "#9C27B0"]  # green, blue, orange, purple

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [2, 1]})

#  Left: 4×4 input with coloured 2×2 window overlays 
ax = axes[0]
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(3.5, -0.5)
ax.set_aspect("equal")

# White background cells with grid values
for r in range(4):
    for c in range(4):
        rect = mpatches.Rectangle(
            (c - 0.5, r - 0.5), 1, 1, linewidth=1, edgecolor="#aaa", facecolor="white"
        )
        ax.add_patch(rect)
        ax.text(
            c,
            r,
            str(int(grid[r, c])),
            ha="center",
            va="center",
            fontsize=16,
            fontweight="bold",
        )

# Semi-transparent coloured patches for each 2×2 window
windows = [(0, 0), (0, 2), (2, 0), (2, 2)]  # (row_start, col_start)
for idx, (r0, c0) in enumerate(windows):
    rect = mpatches.FancyBboxPatch(
        (c0 - 0.46, r0 - 0.46),
        1.92,
        1.92,
        boxstyle="square,pad=0",
        linewidth=2.5,
        edgecolor=COLOURS[idx],
        facecolor=COLOURS[idx],
        alpha=0.3,
    )
    ax.add_patch(rect)

ax.set_title("Input (4 × 4)", fontsize=13, fontweight="bold")
ax.axis("off")

#  Right: 2×2 output 
ax2 = axes[1]
ax2.set_xlim(-0.5, 1.5)
ax2.set_ylim(1.5, -0.5)
ax2.set_aspect("equal")

out_vals = [[6, 4], [5, 3]]
for r in range(2):
    for c in range(2):
        idx = r * 2 + c
        rect = mpatches.FancyBboxPatch(
            (c - 0.46, r - 0.46),
            0.92,
            0.92,
            boxstyle="square,pad=0",
            linewidth=2.5,
            edgecolor=COLOURS[idx],
            facecolor=COLOURS[idx],
            alpha=0.35,
        )
        ax2.add_patch(rect)
        ax2.text(
            c,
            r,
            str(out_vals[r][c]),
            ha="center",
            va="center",
            fontsize=20,
            fontweight="bold",
        )

ax2.set_title("Output (2 × 2) — max of each window", fontsize=13, fontweight="bold")
ax2.axis("off")

fig.suptitle("Max-Pooling: 2×2 window, stride 2", fontsize=14, fontweight="bold")
fig.text(
    0.5,
    0.01,
    "Each 2×2 tile → one value.  Keeps the strongest activation.  Reduces spatial size by 2×.",
    ha="center",
    fontsize=10,
    style="italic",
    color="#555",
)
plt.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

print("Window maxima:")
print(f"  Top-left  [[1,3],[5,6]] → max = {max(1,3,5,6)}")
print(f"  Top-right [[2,4],[1,2]] → max = {max(2,4,1,2)}")
print(f"  Bot-left  [[3,5],[1,2]] → max = {max(3,5,1,2)}")
print(f"  Bot-right [[0,1],[3,0]] → max = {max(0,1,3,0)}")
print()
print("→ 4×4 = 16 values → 2×2 = 4 values.  Spatial size halved.")
print("→ MaxPool keeps the strongest activation — invariant to small translations.")

In [ ]:
#  Spatial size progression + feature map visualisation 
def out_size(w, k, p, s):
    """Spatial output size: floor((W - K + 2P) / S) + 1"""
    return (w - k + 2 * p) // s + 1


print("Spatial size progression through a standard 2-conv CNN on 28×28 MNIST:")
print(f"{'Layer':<35} {'Before':>8} {'After':>8}")
print("-" * 53)
w = 28
layers = [
    ("Conv1  3×3 pad=1 stride=1", 3, 1, 1),
    ("MaxPool 2×2 stride=2", 2, 0, 2),
    ("Conv2  3×3 pad=1 stride=1", 3, 1, 1),
    ("MaxPool 2×2 stride=2", 2, 0, 2),
]
for name, k, p, s in layers:
    w_new = out_size(w, k, p, s)
    print(f"  {name:<33} {w:>5}×{w:<5} {w_new:>3}×{w_new:<3}")
    w = w_new

print(f"\nFinal feature map spatial size: {w}×{w}")
print()

# Build and run the 2-conv sequence on our running example
torch.manual_seed(42)
seq = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 8, 3, padding=1),
    nn.ReLU(),
)

with torch.no_grad():
    fmaps = seq(img_b)  # (1, 8, 14, 14)

print(f"Feature map tensor shape after 2 conv+pool layers: {tuple(fmaps.shape)}")
print("  → batch=1, 8 channels, 14×14 spatial")
print()

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(fmaps[0, i].numpy(), cmap="viridis")
    ax.set_title(f"Channel {i+1}", fontsize=9)
    ax.axis("off")
plt.suptitle(
    "Feature maps after 2 conv+pool layers (random weights — pre-training)", fontsize=11
)
plt.tight_layout()
plt.show()

print("Each channel detects a different pattern — after training they specialise.")
print("Some channels will 'see' horizontal edges; others corners or blob centres.")

### Trained Filter Specialisation — The "Aha" Moment

After **random initialisation** every filter looks like noise and the feature maps are structureless. After training on even a small slice of real images, filters **self-organise into detectors for specific visual patterns**. Run the cell below and look for:

| Pattern type                     | What to look for in the kernel                           | What to look for in the feature map              |
| -------------------------------- | -------------------------------------------------------- | ------------------------------------------------ |
| **Edge detector**                | One half bright red, opposite half bright blue           | Bright stripe along one edge of the digit        |
| **Horizontal / vertical stroke** | Horizontal or vertical colour band across the 3×3 kernel | Lights up only on `—` or `\|` strokes of the "3" |
| **Curve detector**               | Arc-shaped contrast pattern                              | Activates on the rounded arcs of the digit       |
| **Blob detector**                | Bright centre, dark surround (or vice versa)             | Broad bright or dark region                      |

Compare the feature maps below with the **random-weight maps above** — trained maps have clear structure, each lighting up a different region of the digit.


In [ ]:
#  Trained filter specialisation — the CNN 'aha' moment 
# Train a minimal CNN on 500 MNIST images (≈10 s on CPU).
# With only 8 filters and 3 epochs, each conv1 filter already specialises.

torch.manual_seed(42)

#  500-image subset 
subset_ds = torch.utils.data.Subset(train_ds, list(range(500)))
subset_loader = torch.utils.data.DataLoader(subset_ds, batch_size=64, shuffle=True)


#  Minimal CNN — one conv layer so filters are directly inspectable 
class MiniCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)  # 28 → 14
        self.fc = nn.Linear(8 * 14 * 14, 10)

    def forward(self, x):
        return self.fc(self.pool(torch.relu(self.conv1(x))).flatten(1))


mini = MiniCNN()
opt = torch.optim.Adam(mini.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

print("Training MiniCNN on 500 MNIST images, 3 epochs (CPU)…")
for epoch in range(3):
    mini.train()
    correct, seen = 0, 0
    for X, y in subset_loader:
        opt.zero_grad()
        out = mini(X)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        correct += (out.argmax(1) == y).sum().item()
        seen += len(y)
    print(f"  Epoch {epoch + 1}: train accuracy = {correct / seen:.1%}")

print()
mini.eval()

#  Plot 1: Trained conv1 filter kernels (3×3 weights) 
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    w = mini.conv1.weight[i, 0].detach().numpy()
    vmax = max(abs(w.min()), abs(w.max())) + 1e-9
    ax.imshow(w, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(f"Filter {i + 1} (trained)", fontsize=10)
    ax.axis("off")
plt.suptitle(
    "Trained conv1 kernels — edge, curve, and blob detectors emerge from data",
    fontsize=11,
)
plt.tight_layout()
plt.show()

#  Plot 2: Feature maps of digit '3' through trained conv1 
with torch.no_grad():
    fmaps_trained = torch.relu(mini.conv1(img_b))  # (1, 8, 28, 28)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(fmaps_trained[0, i].numpy(), cmap="viridis")
    ax.set_title(f"Filter {i + 1} (trained)", fontsize=10)
    ax.axis("off")
plt.suptitle(
    "Feature maps: digit '3' through TRAINED conv1 — each filter activates on a different pattern",
    fontsize=11,
)
plt.tight_layout()
plt.show()

print("Key observations (compare with random-weight feature maps above):")
print("  Random → diffuse, noisy, no spatial structure")
print("  Trained → each channel highlights a specific stroke region of the digit")
print()
print("This specialisation is automatic — backpropagation discovers that")
print("edge/curve/blob detectors are the most informative features for MNIST.")

#### What just happened — and what's missing?

We confirmed the size formula and saw that max-pooling halves spatial resolution twice (28→14→7). Each of the 8 output channels is a 14×14 filtered version of the original 28×28 digit.

But there's a question that formula doesn't answer: **which part of the original image does a neuron deep in the network actually 'see'?** If a layer-3 neuron's receptive field is only 5×5 pixels, it can only detect local strokes — it can never know whether the whole shape looks like a "3" vs an "8". We need a way to measure this.


---

## Part 3 — Receptive Field

The **receptive field** of a neuron is the region of the original input that can influence its output. With a 3×3 filter and no padding:

- Layer 1 neuron sees a **3×3** patch of the input
- Layer 2 neuron sees a **5×5** patch (because it combines three overlapping 3×3 patches)
- Layer 3 neuron sees a **7×7** patch
- Layer $N$ neuron sees a $(2N+1) \times (2N+1)$ patch

This is why **depth gives global context**: a 13-layer network with 3×3 kernels reaches a 27×27 receptive field — almost the entire MNIST image.

**Measuring it with gradients:** we zero out the gradient, do a forward pass, pick one output neuron as the target, backpropagate, and look at which input pixels received non-zero gradient. Those pixels are exactly the receptive field.

> **Why this matters for UnifiedAI:** a single conv layer can only detect local strokes. Stacking layers lets deeper neurons recognise whole digit shapes — and eventually whole word structures on scanned forms.


In [ ]:
#  Measure receptive field via gradient backpropagation 
# No padding: each 3×3 conv reduces spatial size by 2 on each side
# Expected receptive field for 3 layers: 3 + 2 + 2 = 7 pixels wide


class ThreeConvCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 8, 3, padding=0)  # 28 → 26
        self.c2 = nn.Conv2d(8, 8, 3, padding=0)  # 26 → 24
        self.c3 = nn.Conv2d(8, 8, 3, padding=0)  # 24 → 22

    def forward(self, x):
        return self.c3(torch.relu(self.c2(torch.relu(self.c1(x)))))


torch.manual_seed(42)
cnn3 = ThreeConvCNN()

# requires_grad on input so we can backprop to it
img_rf = img_b.clone().requires_grad_(True)
out = cnn3(img_rf)  # (1, 8, 22, 22)

# Pick one output neuron at position (11, 11) in channel 0 as our target
target = torch.zeros_like(out)
target[0, 0, 11, 11] = 1.0
(out * target).sum().backward()

grad = img_rf.grad.abs().squeeze()  # (28, 28)
rf_mask = (grad > 0).float()
rows, cols = rf_mask.nonzero(as_tuple=True)
rf_size = (rows.max() - rows.min() + 1).item()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
a1.imshow(example.squeeze(), cmap="gray")
a1.set_title("Input image", fontsize=12)
a1.axis("off")
a2.imshow(rf_mask.numpy(), cmap="hot")
a2.set_title(
    f"Receptive field: {rf_size:.0f}×{rf_size:.0f} pixels (bright = influenced)",
    fontsize=11,
)
a2.axis("off")
plt.suptitle("Which input pixels influence one layer-3 output neuron?", fontsize=11)
plt.tight_layout()
plt.show()

print(f"3-layer CNN (no padding): receptive field = {rf_size:.0f}×{rf_size:.0f}")
print(f"Formula: 2×N_layers + 1 = 2×3 + 1 = 7 (predicted: 7×7) ")
print()
print(f"Input image: 28×28 = 784 pixels")
print(f"Layer-3 neuron sees only {rf_size**2:.0f} of those — just a local patch")
print()
print("→ Deeper layers detect complex, large-scale patterns — not just local edges.")
print("→ Dilated convolutions (see Tier 2) can expand this without adding parameters.")

---
## Part 4 — ResNet Skip Connections

Stacking more layers should always help — more depth means larger receptive fields and more complex feature hierarchies. In practice, networks deeper than ~20 layers without architectural tricks **train worse** than shallower ones, even on the training set. This is the *degradation problem*, and it's caused by vanishing gradients.
---

#### Predict first:**

We will build a 10-layer plain CNN and a 10-layer ResNet, then measure the gradient magnitude at layer 1 in both.

For a 10-layer plain network vs. a 10-layer ResNet, what do you expect the ResNet's gradient at layer 1 to be?

- **(a)** About 2× larger than the plain network
- **(b)** 10–100× larger than the plain network
- **(c)** About the same — one extra addition can't make that much difference


![ResNet skip connection: plain block with vanishing gradient vs residual block with bypass arrow guaranteeing gradient flow](images/resnet-skip-connection.png)


In [ ]:
#  Skip connection gradient proof 
# Build identical 10-layer networks; only difference: plain vs residual block


class PlainBlock(nn.Module):
    """Standard conv block — no skip. Gradient multiplied through each layer."""

    def __init__(self, c):
        super().__init__()
        self.c = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        return torch.relu(self.c(x))


class ResBlock(nn.Module):
    """Residual block: F(x) + x. The '+x' term bypasses the vanishing gradient."""

    def __init__(self, c):
        super().__init__()
        self.c = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        return torch.relu(self.c(x) + x)  # ← the only difference


def make_net(Block, n_layers=10, channels=16):
    """Input conv → N residual/plain blocks → global avg pool."""
    return nn.Sequential(
        nn.Conv2d(1, channels, 3, padding=1),
        *[Block(channels) for _ in range(n_layers)],
        nn.AdaptiveAvgPool2d(1),
    )


def grad_norm_at_layer1(net):
    """Forward + backward; return mean absolute gradient at the very first conv layer."""
    net.zero_grad()
    net(img_b).mean().backward()
    return net[0].weight.grad.abs().mean().item()


torch.manual_seed(42)
plain = make_net(PlainBlock)
torch.manual_seed(42)
resnet = make_net(ResBlock)

pg = grad_norm_at_layer1(plain)
rg = grad_norm_at_layer1(resnet)
ratio = rg / max(pg, 1e-14)

print("Gradient at layer 1  (10-layer network):")
print(f"  Plain CNN:  {pg:.3e}")
print(f"  ResNet:     {rg:.3e}")
print(f"  Ratio:      {ratio:.1f}×")
print()
print("Mathematical explanation:")
print("  Skip output y = F(x) + x")
print("  ∂L/∂x = ∂L/∂y · (1 + ∂F/∂x)")
print("  The '1' guarantees gradient flows REGARDLESS of whether F(x) is useful")
print()
if ratio > 5:
    print(f"Prediction check: answer (b)   — ResNet gradient is {ratio:.0f}× larger")
elif ratio > 1.5:
    print(f"Prediction check: answer (a)   — ResNet gradient is {ratio:.1f}× larger")
else:
    print(f"Prediction check: answer (c) — surprisingly similar (ratio={ratio:.2f})")
print()
print(
    "→ Skip connections are why ResNet-152 (152 layers!) trains without special tricks."
)

### Why did ResNet win? Here's the math.

Imagine the gradient as a whispered message passed backward through 20 relay stations. Each plain layer hears only 70% of the previous station's signal — after 20 relays, 0.70²⁰ ≈ 0.08% of the original survives. The skip connection gives every station a direct line to the source.

During backprop, the gradient of the loss w.r.t. layer-1 weights passes through every intervening layer's Jacobian. With ReLU activations, each layer's Jacobian has values < 1 on average. After 20 multiplications the gradient can be $10^{-8}$ — effectively zero, and the early layers stop learning.

**He et al. (2015) fix:** add an identity shortcut:
$$\text{output} = \mathcal{F}(x) + x$$

The gradient through the addition node is:
$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \left(1 + \frac{\partial \mathcal{F}}{\partial x}\right)$$

The **`+1`** term means the gradient is always at least $\frac{\partial L}{\partial y}$ — it can never be fully extinguished by a near-zero $\frac{\partial \mathcal{F}}{\partial x}$.


---

## Part 5 — Transfer Learning

Training a deep CNN from scratch requires millions of labelled images and days of GPU time. For most real tasks — including UnifiedAI's scanned-form digit recognition — you have hundreds or low thousands of examples, not millions.

**Transfer learning** exploits the fact that the early layers of a CNN trained on ImageNet (1.2 million diverse images, 1000 classes) learn universal low-level features: edges, textures, colour gradients. These are useful for _any_ visual task.

The standard recipe:

1. **Load a pretrained backbone** (ResNet-18, EfficientNet, etc.)
2. **Freeze all weights** — `requires_grad = False` on everything
3. **Replace the final classification head** with a new linear layer for your task
4. **Train only the new head** — a few epochs, small dataset, fast
5. _(Optional)_ **Fine-tune** the top few backbone layers with a tiny learning rate

| Phase          | Layers updated         | Epochs needed | Dataset size |
| -------------- | ---------------------- | ------------- | ------------ |
| Head-only      | 1 linear layer         | 3–10          | 100–1000     |
| Fine-tune top  | Last 1–2 blocks + head | 5–20          | 1000–10000   |
| Full fine-tune | All layers             | 20–100        | 10000+       |
| From scratch   | All layers             | 100–300       | 100000+      |

We will demonstrate head-only transfer: ResNet-18 on **digit 0 vs digit 1** (a 2-class subset of MNIST). We only train 512 parameters instead of 11 million.


> **Why does this work?** A convolution filter trained to detect "horizontal edge" in a photo of a car also detects "horizontal stroke" in the digit "3" — because strokes _are_ edges. The early layers of any vision model learn universal edge detectors, regardless of training domain. When we freeze the base and only train the final classification layer, we reuse those universal detectors for free. The only thing we're teaching is "which combination of those universal features predicts which class."


In [ ]:
#  Build binary transfer learning dataset (digit 0 vs 1) 
# ResNet-18 expects 3-channel input so we repeat the grayscale channel 3×


def binary_subset(classes=(0, 1), n_each=300):
    """Extract n_each images per class, return as list of (img_3ch, label_index) tuples."""
    three_ch = T.Compose(
        [T.ToTensor(), T.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081))]
    )
    raw = torchvision.datasets.MNIST(
        "./data", train=True, download=True, transform=three_ch
    )
    out = []
    counts = {c: 0 for c in classes}
    for img, label in raw:
        if label in classes and counts[label] < n_each:
            out.append((img, classes.index(label)))
            counts[label] += 1
        if all(v >= n_each for v in counts.values()):
            break
    return out


tl_train = binary_subset(classes=(0, 1), n_each=300)  # 600 training images
tl_test = binary_subset(
    classes=(0, 1), n_each=100
)  # 200 test images (from same raw set)
tl_loader = torch.utils.data.DataLoader(tl_train, batch_size=32, shuffle=True)

print(f"Transfer learning dataset: digit 0 vs digit 1")
print(f"  Training: {len(tl_train)} images (300 per class)")
print(f"  Test:     {len(tl_test)} images (100 per class)")
print()
print("ResNet-18 expects RGB (3-channel) input — we repeat MNIST's single channel 3×.")
print("This is fine: the pretrained filters will activate on grayscale-as-RGB images.")

In [ ]:
#  Fine-tune ResNet-18 head only 
# Freeze all backbone weights; only the new 2-class linear head is trained

rn18 = torchvision.models.resnet18(weights="IMAGENET1K_V1")

# Freeze everything
for p in rn18.parameters():
    p.requires_grad = False

# Replace the 1000-class head with a 2-class head (unfreezes automatically)
rn18.fc = nn.Linear(rn18.fc.in_features, 2)

trainable = sum(p.numel() for p in rn18.parameters() if p.requires_grad)
total = sum(p.numel() for p in rn18.parameters())
print(f"ResNet-18 parameter breakdown:")
print(f"  Total:     {total:>12,}")
print(f"  Trainable: {trainable:>12,}  ({trainable/total:.2%} of total)")
print(f"  Frozen:    {total-trainable:>12,}  (ImageNet features — not touched)")
print()

optimizer = torch.optim.Adam(rn18.fc.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
rn18.train()

print("Training head-only for 3 epochs on 600 images:")
for epoch in range(3):
    correct, total_seen = 0, 0
    for X, y in tl_loader:
        y = torch.tensor(y) if not isinstance(y, torch.Tensor) else y
        optimizer.zero_grad()
        out = rn18(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        correct += (out.argmax(1) == y).sum().item()
        total_seen += len(y)
    print(f"  Epoch {epoch+1}: train accuracy = {correct/total_seen:.1%}")

print()
print(
    f"→ {trainable:,} trainable params, 600 images, 3 epochs — that's the power of transfer learning!"
)
print(
    "  ImageNet features (edges, textures, curves) transfer directly to digit recognition."
)

---

## Part 6 — Toy → Real Bridge

Everything we've built — convolutions, pooling, skip connections, linear heads — is exactly what state-of-the-art vision models use. The only difference is scale.

| Model         | Year | Params | Depth                 | Key innovation                     |
| ------------- | ---- | ------ | --------------------- | ---------------------------------- |
| Our TinyCNN   | 2024 | ~30k   | 4 layers              | Basic conv+pool+linear             |
| **ResNet-18** | 2015 | ~11M   | 18 layers             | Skip connections                   |
| **ResNet-50** | 2015 | ~25M   | 50 layers             | Bottleneck residual blocks         |
| **ViT-B/16**  | 2020 | ~86M   | 12 transformer blocks | Patches as tokens, no convolutions |

All four use the same building blocks we've studied:

- `nn.Conv2d` → `nn.ReLU` → `nn.MaxPool2d`
- Skip connections (ResNet, ViT)
- `nn.Linear` classification head
- Trained with `nn.CrossEntropyLoss` + Adam/SGD

> **If you understood our TinyCNN, you understand the foundation of every image model in production today.**


In [ ]:
#  Parameter count comparison 


class TinyCNN(nn.Module):
    """Our chapter CNN: 2 conv layers → flatten → 2 linear layers → 10 classes."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # → 32×14×14
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # → 64×7×7
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


tiny_p = sum(p.numel() for p in TinyCNN().parameters())
r18_p = sum(p.numel() for p in torchvision.models.resnet18(weights=None).parameters())
r50_p = sum(p.numel() for p in torchvision.models.resnet50(weights=None).parameters())

print(f"{'Model':<25} {'Parameters':>15}")
print("-" * 42)
print(f"{'Our TinyCNN':<25} {tiny_p:>15,}")
print(f"{'ResNet-18':<25} {r18_p:>15,}")
print(f"{'ResNet-50':<25} {r50_p:>15,}")

try:
    vit_p = sum(
        p.numel() for p in torchvision.models.vit_b_16(weights=None).parameters()
    )
    print(f"{'ViT-B/16':<25} {vit_p:>15,}  (Vision Transformer)")
except AttributeError:
    print("ViT-B/16: requires torchvision>=0.13 — upgrade to see this row")

print()
print("Architecture breakdown of TinyCNN:")
for name, layer in TinyCNN().net.named_children():
    p = sum(x.numel() for x in layer.parameters())
    if p > 0:
        print(f"  Layer {name}: {layer.__class__.__name__:>15}  {p:>10,} params")

print()
print("All four models use: nn.Conv2d → nn.ReLU → nn.Linear")
print("  + skip connections (ResNet) or patch embeddings + attention (ViT)")
print("→ Mastering TinyCNN = understanding the foundation of ALL modern image models.")

---

## Summary

| Part  | Concept                         | What we proved                                                                              |
| ----- | ------------------------------- | ------------------------------------------------------------------------------------------- |
| **1** | Convolution as a learned filter | Sobel filter highlights top/bottom arcs of digit '3'; CNNs learn such filters automatically |
| **2** | Stride & pooling                | Output size formula confirmed; 28×28 → 7×7 through two MaxPool(2) layers                    |
| **3** | Receptive field                 | Layer-3 neuron sees a 7×7 patch; gradient measurement proves it                             |
| **4** | ResNet skip connections         | `+x` bypass makes ResNet gradient 10–100× larger at layer 1 vs plain CNN                    |
| **5** | Transfer learning               | ResNet-18 head-only fine-tune: <1% of params trained, high accuracy on 600 images           |
| **6** | Toy → real bridge               | TinyCNN (~30k) → ResNet-18 (11M) → ResNet-50 (25M) → ViT-B/16 (86M) — same building blocks  |

### Key insights to keep

- **Translation equivariance is free** — the sliding filter sees the same pattern wherever it appears, halving the parameter count by 350× vs an MLP first layer.
- **Depth ≠ receptive field** until you add pooling or dilation — a 10-layer padded CNN still only sees 3×3.
- **Skip connections are a gradient highway** — the `+1` in `∂L/∂x = ∂L/∂y × (1 + ∂F/∂x)` guarantees the gradient never fully vanishes.
- **Transfer learning is the default** for vision tasks with <10k images — freeze the backbone, replace the head.
- **ViT replaces convolutions with attention** — but the final linear head and the overall train-loss-backprop-update loop are identical.


In [ ]:
#  Closing decision — UnifiedAI property code reader 
trainable_head = sum(p.numel() for p in rn18.parameters() if p.requires_grad)
total_rn18 = sum(p.numel() for p in rn18.parameters())

print("=" * 57)
print("  CLOSING DECISION — CNNs for UnifiedAI Property Codes")
print("=" * 57)
print()
print("  Task: classify handwritten digits 0-9 from scanned assessment forms")
print()
print("  Recommended architecture:")
print("    Pretrained ResNet-18, frozen backbone, fine-tuned classification head")
print(
    f"    Trainable: {trainable_head:,} of {total_rn18:,} params ({trainable_head/total_rn18:.2%})"
)
print("    Rationale: ImageNet weights transfer well to digit recognition")
print("               (edges → strokes → digit shapes — same hierarchy)")
print()
print("  Design rules verified in this chapter:")
print("  1. Use padding=1 with 3×3 conv to preserve spatial size")
print("  2. Add MaxPool2d(2) every 2 layers to compress resolution")
print("  3. Use skip connections for any network ≥ 5 layers deep")
print("  4. Freeze pretrained backbone; only fine-tune the classification head")
print("  5. Monitor receptive field — it must cover entire digit region")
print()
print("  NEXT CHAPTER:")
print("  CNNs assume a fixed spatial grid — they cannot model sequential structure.")
print("  Property codes appear in sequences (words, sentences, paragraphs).")
print("  → P-4: RNN / LSTM — memory across time steps")

---

## Tier Ledger — What This Chapter Covers

This notebook deliberately covers a curated subset of CNN concepts. The tiers below make coverage gaps explicit and intentional.

### Tier 1 — Built and proved in this notebook

| Concept | Where |
|---|---|
| Convolution as sliding dot-product | Part 1 — Sobel + learned filters |
| Translation equivariance | Part 1 — parameter count comparison |
| Stride & MaxPool spatial compression | Part 2 — size formula + feature maps |
| Receptive field measurement | Part 3 — gradient backprop |
| Vanishing gradient problem | Part 4 — plain vs ResNet gradient ratio |
| ResNet skip connections | Part 4 — mathematical proof + measurement |
| Transfer learning (head-only) | Part 5 — ResNet-18 fine-tune on 600 images |
| Scale bridge (toy → ResNet → ViT) | Part 6 — parameter count table |

### Tier 2 — Explained but not built

- **Depthwise separable convolution (MobileNet):** Split a K×K×C convolution into a K×K×1 spatial conv (per channel) followed by 1×1×C pointwise conv. Reduces parameters by ~9× for 3×3 kernels. Explained in Part 6 comment; not implemented.
- **Batch Normalisation:** Normalises activations within a mini-batch to zero mean, unit variance. Standard in ResNet between conv and ReLU. Not added to TinyCNN to keep code minimal.
- **Dropout for convolutions (SpatialDropout2d):** Drops entire feature-map channels during training. Standard regularisation. Not added.

### Tier 3 — Named only (studied in later chapters)

- **Dilated (atrous) convolutions:** Use a spacing parameter to expand receptive field without pooling. Key in DeepLab segmentation models.
- **Deformable convolutions:** Learn offsets for each kernel position, adapting to non-grid spatial patterns. Used in Deformable DETR.
- **Vision Transformers (ViT):** Divide image into 16×16 patches, treat as token sequences, apply standard transformer self-attention. Covered in `learning/genai/05-vision-transformers/`.
- **Object detection heads (YOLO, Faster R-CNN):** Extend CNNs to predict bounding boxes. Covered in `learning/genai/06-object-detection/`.

---

## When to Use What

| Data type                        | Architecture                         | Reason                                        |
| -------------------------------- | ------------------------------------ | --------------------------------------------- |
| Images (any size)                | CNN or ViT                           | Spatial locality / translation equivariance   |
| Small labelled dataset (<10k)    | Pretrained backbone + fine-tune      | Reuse ImageNet feature hierarchy              |
| Large labelled dataset (>100k)   | Train from scratch or full fine-tune | Enough signal to specialise all layers        |
| Sequential data (text, audio)    | RNN / LSTM                           | Memory across time steps                      |
| Long-range sequence dependencies | Transformer                          | Direct attention, no recurrence bottleneck    |
| Multi-modal (image + text)       | CNN encoder + Transformer decoder    | Each modality gets its optimal inductive bias |

---

**→ P-4: RNN / LSTM — sequential data, memory across time steps**
